# LAB 4: Forecasting & Predictive Analytics cho dữ liệu IoT

Bài này dùng UCI Appliances Energy Prediction để dự báo `Appliances` trong bước thời gian kế tiếp. Điểm cần nhớ: Lab 4 không hỏi điểm hiện tại có bất thường không; Lab 4 hỏi giá trị tương lai có thể là bao nhiêu và quyết định vận hành nên thận trọng thế nào.

## 1. Load dữ liệu
Chạy `src/download_data.py` trước nếu muốn tải UCI official dataset. Nếu không có Internet, notebook vẫn chạy bằng sample fallback.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
from utils import load_dataset, make_supervised_frame, clean_supervised_frame, FEATURE_COLUMNS, time_split, regression_metrics, risk_from_prediction, recommendation_from_risk
import pandas as pd
import numpy as np

df = load_dataset()
df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [2]:
df[['date','Appliances','lights','T1','RH_1','T_out']].describe()

,date,Appliances,lights,T1,RH_1,T_out
count,19735,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000
mean,2016-03-20 05:29:59.999999,97.694958,3.801875,21.686571,40.259739,7.411665
min,2016-01-11 17:00:00,10.000000,0.000000,16.790000,27.023333,-5.000000
25%,2016-02-14 23:15:00,50.000000,0.000000,20.760000,37.333333,3.666667
50%,2016-03-20 05:30:00,60.000000,0.000000,21.600000,39.656667,6.916667
75%,2016-04-23 11:45:00,100.000000,0.000000,22.600000,43.066667,10.408333
max,2016-05-27 18:00:00,1080.000000,70.000000,26.260000,63.360000,26.100000
std,NaN,102.524891,7.935988,1.606066,3.979299,5.317409


## 2. Tạo supervised forecasting table
Ta tạo target tương lai bằng `target_future = Appliances.shift(-1)`. Với UCI, một bước là 10 phút.

In [3]:
supervised = make_supervised_frame(df, horizon_steps=1, include_target=True)
supervised = clean_supervised_frame(supervised, FEATURE_COLUMNS, require_target=True)
supervised[['date','Appliances','appliances_lag_1','appliances_rolling_mean_6','target_future']].head()

,date,Appliances,appliances_lag_1,appliances_rolling_mean_6,target_future
0,2016-01-11 21:00:00,110,110.0,130.000000,110.0
1,2016-01-11 21:10:00,110,110.0,125.000000,100.0
2,2016-01-11 21:20:00,100,110.0,121.666667,100.0
3,2016-01-11 21:30:00,100,100.0,106.666667,100.0
4,2016-01-11 21:40:00,100,100.0,105.000000,100.0


## 3. Chia train/test theo thời gian
Không dùng random split cho chuỗi thời gian vì sẽ làm thông tin tương lai rò rỉ ngược về quá khứ.

In [4]:
train_df, test_df = time_split(supervised, train_ratio=0.75)
train_df['date'].min(), train_df['date'].max(), test_df['date'].min(), test_df['date'].max()

(Timestamp('2016-01-11 21:00:00'),
 Timestamp('2016-04-23 12:30:00'),
 Timestamp('2016-04-23 12:40:00'),
 Timestamp('2016-05-27 17:50:00'))

## 4. Baseline: Last Value và Moving Average
Baseline là hàng rào chống ảo tưởng. Model ML chỉ có ý nghĩa khi được so với quy tắc đơn giản.

In [5]:
y_test = test_df['target_future']
last_value_pred = test_df['Appliances']
ma6_pred = test_df['appliances_rolling_mean_6']
print('Last value:', regression_metrics(y_test, last_value_pred))
print('Moving average 6:', regression_metrics(y_test, ma6_pred))

Last value: {'mae': 25.7812, 'rmse': 64.4874, 'mape_percent': 21.8438, 'forecast_bias': -0.067}
Moving average 6: {'mae': 33.7003, 'rmse': 74.0469, 'mape_percent': 29.496, 'forecast_bias': -0.0822}


## 5. Train Linear Regression và Random Forest
Linear Regression dễ giải thích; Random Forest mạnh hơn nhưng cần tránh coi nó là hộp thần kỳ.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['target_future']
X_test = test_df[FEATURE_COLUMNS]

lin = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
lin.fit(X_train, y_train)
lin_pred = lin.predict(X_test)
print('Linear Regression:', regression_metrics(y_test, lin_pred))

rf = RandomForestRegressor(n_estimators=120, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print('Random Forest:', regression_metrics(y_test, rf_pred))

Linear Regression: {'mae': 27.4157, 'rmse': 58.1258, 'mape_percent': 27.2872, 'forecast_bias': 2.5689}


Random Forest: {'mae': 40.1542, 'rmse': 71.6956, 'mape_percent': 41.6192, 'forecast_bias': 22.4595}


## 6. Model nâng cao: GradientBoostingRegressor
Đây là bản demo nâng cao nhẹ theo hướng boosting. Nó là cầu nối để sinh viên hiểu XGBoost/LightGBM nhưng vẫn chạy bằng scikit-learn, không cần cài thêm thư viện nặng.

In [7]:
hgb = GradientBoostingRegressor(n_estimators=160, learning_rate=0.05, max_depth=3, min_samples_leaf=3, random_state=42)
hgb.fit(X_train, y_train)
hgb_pred = hgb.predict(X_test)
print('Gradient Boosting:', regression_metrics(y_test, hgb_pred))

Gradient Boosting: {'mae': 39.469, 'rmse': 66.3544, 'mape_percent': 45.8698, 'forecast_bias': 21.3498}


## 7. Từ forecast sang risk_level và recommendation
Forecast chưa phải quyết định điều khiển. Dự báo cần đi qua risk rule và safety rule.

In [8]:
thresholds = {'warning': float(np.quantile(y_train, 0.70)), 'high': float(np.quantile(y_train, 0.90)), 'critical': float(np.quantile(y_train, 0.97))}
example_pred = float(hgb_pred[-1])
risk = risk_from_prediction(example_pred, thresholds)
print(example_pred, risk, recommendation_from_risk(risk), thresholds)

436.1253066808061 CRITICAL HUMAN_CHECK_BEFORE_ACTUATOR_CONTROL {'warning': 90.0, 'high': 210.0, 'critical': 400.0}


## 8. Chạy script tổng hợp
Sau khi hiểu notebook, sinh viên chạy:

```bash
python src/train_forecast.py
python src/plot_results.py
python src/test_api_local.py
```